In [ ]:
from typing import Iterable, Callable
from itertools import chain as iterchain, combinations as itercomb, product as iterprod
from collections import deque, Counter


iterflat = iterchain.from_iterable

In [ ]:
from board import DIGITS, Loc, Cell, Board
from analytics import Zone, Target
from analytics import visibility, allvisible
from utils import count_finals, draftborhood, draftboard, validate
from solving import solver, Resolution, Resolving, Resolver, solve_logging

In [ ]:
def filt_finals(c: Cell):
    return c.is_final


def filt_drafts(c: Cell):
    return c.is_draft


def filt_having(d: int):
    def filt(c: Cell):
        return d in c.dgs

    return filt


def flat_cells(cc: Iterable[Cell]):
    return iterflat(c.dgs for c in cc)

## Basic

Singles in localities and their contra-neighbors


In [ ]:
def cleanup(board: Board) -> Resolving:
    """Removing drafts contradicting with neighbouring finals"""

    for fincell in filter(filt_finals, iter(board)):
        findig = fincell.final
        assert findig is not None
        for zone in Zone.around(Zone.of(fincell)):
            spoilers = tuple(filter(filt_having(findig), draftborhood(board, zone)))
            if len(spoilers):
                yield Resolution(
                    castaways={Target.to(c, findig) for c in spoilers},
                    highlights={"anchors": {Target.to(fincell, findig)}, "zone": zone},
                )

In [ ]:
def singles(board: Board) -> Resolving:
    """Isolate singular digits in localities"""

    for zone in Zone.All():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            if cnt == 1:
                (lonesome,) = filter(filt_having(dig), drafts)
                yield Resolution(
                    finals={Target.to(lonesome, dig)},
                    highlights={"zone": zone, "anchors": {Target(zone, dig)}},
                )

## Multiples

Combos of N digits


#### open

Some n cells (within locality) contains only n-combo // the digits may be in other cells

Rule: remove the digits of the combo from all other cells


In [ ]:
def clean_mults(board: Board, mult: int) -> Resolving:
    """Clean out spoiled neighbours of open multiples in each zone"""
    for zone in Zone.All():
        drafts = set(draftborhood(board, zone))
        inhabitants = set(flat_cells(drafts))
        for cmb in itercomb(inhabitants, mult):
            combo = set(cmb)
            # all cells containing only the combo
            habitat = set(filter(lambda c: c.dgs <= combo, drafts))
            # all other neighbors containing some combo digits
            spoilers = tuple(filter(lambda c: c.dgs & combo, drafts - habitat))

            if len(habitat) == mult and len(spoilers):
                yield Resolution(
                    castaways={Target.to(c, d) for c in spoilers for d in c.dgs & combo},
                    highlights={
                        "zone": zone,
                        "anchors": {Target.to(c, d) for c in habitat for d in c.dgs & combo},
                    },
                )


def clean_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return clean_mults(board, mult)

    resolver.__name__ = f"open_mults[{mult}]"
    return resolver

### hidden

Some n-combo contained in only n cells (within locality) // along other drafts

Rule: remove all other drafts from the cells => it becomes open


In [ ]:
def unhide_mults(board: Board, mult: int) -> Resolving:
    """Clean up cellmates of hidden multiples"""
    for zone in Zone.All():
        drafts = set(draftborhood(board, zone))
        inhabitants = set(flat_cells(drafts))
        for cmb in itercomb(inhabitants, mult):
            combo = set(cmb)
            if len(inhabitants & combo) != mult:
                continue
            # all cells containing some combo digits (+ some spoilers)
            habitat = tuple(filter(lambda c: c.dgs & combo, drafts))
            # inhabited cells with other digits
            spoiled = tuple(filter(lambda c: c.dgs - combo, habitat))
            if len(habitat) == mult and len(spoiled):
                yield Resolution(
                    castaways={Target.to(c, d) for c in spoiled for d in c.dgs - combo},
                    highlights={
                        "zone": zone,
                        "anchors": set(Target.to(c, d) for c in habitat for d in c.dgs & combo),
                    },
                )


def unhide_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return unhide_mults(board, mult)

    resolver.__name__ = f"hidden_mults[{mult}]"
    return resolver

## Links

Links represent XOR or NAND relations between drafts.

- XOR $\veebar$ corresponds to "each digit appears only once in a locality"
- NAND $\barwedge$ corrsponds to "each locality contains only different digits"

(or vise versa, I dunno)


In [ ]:
from analytics import Link, HLink, SLink

#### strong/hard links

Represent XOR relation $\veebar$

Criteria:

- only 2 drafts of same digit in a locality
- only 2 drafts (of different digits) in a cell


In [ ]:
def search_hard(board: Board) -> Iterable[HLink]:
    """Scan for all 1:1 hard links in the board"""
    # intra-cellular
    for cell in draftboard(board):
        if len(cell.dgs) == 2:
            d1, d2 = cell.dgs
            yield HLink((Target.to(cell, d1), Target.to(cell, d2)))

    # intra-zonal
    for zone in Zone.All():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            if cnt == 2:
                (n1, n2) = filter(filt_having(dig), drafts)
                yield HLink((Target.to(n1, dig), Target.to(n2, dig)))

#### weak/soft links

Represent NAND relation $\barwedge$

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts (of different digits) in a cell

Note: The criteria are totally independent of board content (calculating from locations only)

Visibility = soft-linkability


In [ ]:
def check_soft(t1: Target, t2: Target):
    assert t1 != t2
    if t1.dig != t2.dig:
        # different digits within a cell
        return t1.is_cellular and t2.is_cellular and t1 == t2
    else:
        # same digits in some shared zone
        return len(visibility(t1.zone, t2.zone)) > 0

## Chains

Alterating link chains constituted of `~ hard ~ soft ~` and `~ soft ~ hard ~`

Lemma1: $(X \barwedge A) \cdot (A \veebar B) \cdot (B \barwedge X) \Rightarrow \neg X$

Meaning: all draft visible (soft-linkable) from some XORed points, are all invalid

Lemma2: $(X \veebar A) \cdot (A \barwedge B) \cdot (B \veebar Y) \Rightarrow (X \veebar Y)$

Meaning: a ALC (of any length) with hard edges behaves as if its edges are hard-linked


In [ ]:
from analytics import Chain

In [ ]:
def search_soft(board: Board, e1: Target, e2: Target) -> Iterable[Target]:
    """Scan for all targets nand-able with both e1 and e2"""
    interest: set[int] = {e1.dig, e2.dig}  # max=2

    for vizone in allvisible(e1.zone, e2.zone):  # max=2
        for cell in filter(lambda c: c.dgs & interest, draftborhood(board, vizone)):  # max=18
            for dig in interest:  # max=36
                trg = Target.to(cell, dig)
                if trg == e1 or trg == e2:
                    continue
                if check_soft(trg, e1) and check_soft(trg, e2):
                    yield trg

#### loop ALC

ALC with connected edges: `X ~ hard ~ ... ~ soft ~ X`

Rule: invalidate all draft visible from each soft-link in the chain


In [ ]:
def match_loop(chain: Chain):
    """ALC loop"""
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def resolve_loop(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible for each soft link"""
    anchors: set[Target] = chain.anchors()  # to exclude linking back to chain

    for link in filter(lambda lnk: isinstance(lnk, SLink), chain):
        t1, t2 = link
        spoilers = set(search_soft(board, t1, t2)) - anchors
        if len(spoilers):
            yield Resolution(
                castaways=spoilers,
                highlights={"anchors": {t1, t2}, "chain": chain},
            )

#### open ALC

ALC with hard links at its edges: `X ~ hard ~ ... ~ hard ~ Y`

Rule: invalidate all drafts visible from both edges of such chain


In [ ]:
def match_rope(chain: Chain):
    """ALC with matching edges"""
    e1, e2 = chain.edges
    return len(chain) > 2 and len(chain) % 2 == 1 and e1.dig == e2.dig


def resolve_rope(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible from both edges"""
    anchors = chain.anchors()
    e1, e2 = chain.edges

    spoilers = set(search_soft(board, e1, e2)) - anchors
    if len(spoilers):
        yield Resolution(
            castaways=spoilers,
            highlights={"anchors": {e1, e2}, "chain": chain},
        )

### Search for chains

- searching for all hard inks first
- trying to connect them into chains


In [ ]:
def expand_alc(chain: Chain, links: Iterable[HLink]) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""
    e1, e2 = chain.edges

    # closing loop
    if len(chain) > 2 and isinstance(chain[0], HLink) and isinstance(chain[-1], HLink) and check_soft(e1, e2):
        yield Chain.extend(chain, SLink((e2, e1)))

    anchors: set[Target] = chain.anchors()

    def noncycling(lnk: Link):
        return lnk[0] not in anchors and lnk[1] not in anchors

    for link in filter(noncycling, links):
        x1, x2 = link
        if check_soft(e2, x1):
            yield Chain.extend(chain, SLink((e2, x1)), link)
        if check_soft(e2, x2):
            yield Chain.extend(chain, SLink((e2, x2)), link.reversed())
        if check_soft(x2, e1):
            yield Chain.extendhead(chain, link, SLink((x2, e1)))
        if check_soft(x1, e1):
            yield Chain.extendhead(chain, link.reversed(), SLink((x1, e1)))


def search_chains_breadth(links: Iterable[HLink], creiteria: Callable[[Chain], bool]) -> Iterable[Chain]:
    """Search for all chain matching creiteria
    breadth-first search (shortest-first)
    it yields infinitely
    breaking search is up to calling coroutine
    """
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # stack
    explored = set[Chain]()
    while frontier:
        chain = frontier.popleft()
        if creiteria(chain):
            yield chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)


def search_chains_depth(links: Iterable[HLink], creiteria: Callable[[Chain], bool]) -> Iterable[Chain]:
    """Search for all chain matching creiteria
    depth-first search (longest-first)
    it yields infinitely
    breaking search is up to calling coroutine
    """
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # stack
    explored = set[Chain]()
    while frontier:
        chain = frontier.pop()
        if creiteria(chain):
            yield chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)

In [ ]:
def chains_(kind: str, strategy: str = "B"):
    if strategy == "D":
        searching = search_chains_depth
    elif strategy == "B":
        searching = search_chains_breadth
    else:
        raise ValueError()

    if kind == "loop":
        matching = match_loop
        resolving = resolve_loop
    elif kind == "rope":
        matching = match_rope
        resolving = resolve_rope
    else:
        raise ValueError()

    def resolver(current: Board) -> Resolving:
        links = set(search_hard(current))

        for chain in searching(links, matching):
            res = tuple(resolving(current, chain))
            if not res:
                # continue to search if if didn't work
                continue

            yield from res
            break

    resolver.__name__ = f"chains[{strategy}][{kind}]"

    return resolver

## A puzzle


In [ ]:
from utils import bparse, fillempty

# totally normal level puzzle:
# solvable by: basics + multiples[2, 3, 4] + chains[rope]
puzzle = bparse("""
....89...
......17.
6........
.2.3.....
.1......9
.......68
8.9.5....
...7..2..
5........
""")
# best result:
# 209 (m2 m3 m4 m5 [B2][rope])

puzzle = Board.transform(puzzle, fillempty)

In [ ]:
puzzle = await solve_logging(
    puzzle,
    cleanup,
    singles,
    clean_mults_(2),
    unhide_mults_(2),
    clean_mults_(3),
    unhide_mults_(3),
    clean_mults_(4),
    unhide_mults_(4),
    clean_mults_(5),
    unhide_mults_(5),
    chains_("rope"),
    chains_("loop"),
    filtout={"cleanup", "singles"},
)

### GUI


In [ ]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-input-color: var(--vscode-editor-foreground);
    --jp-widgets-input-background-color: var(--vscode-editor-background);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.jupyter-widgets input {
   background-color: var(--jp-widgets-input-background-color);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [ ]:
import asyncio
from collections import Counter
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display
from pprint import pprint

from traitlets import HasTraits, Instance, Set, Unicode, observe, Enum, Bool, Dict, Union
from canvas import SudokuCanvas

In [ ]:
# GUI meta-widget


def click_future(button: w.Button) -> asyncio.Future[bool]:
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


LINK_STYLES = {
    "Link": "SOLID",
    "HLink": "HARD",
    "SLink": "SOFT",
}


class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Enum(["INCOMPLETE", "SOLVED", "BROKEN"])
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Target))
    finals = Set(Instance(Target))
    anchors = Set(Instance(Target))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()

        self._hlayers = set()
        self._counters = {dig: self.init_counter(dig) for dig in DIGITS}
        self._counter_total = w.Label(
            "...",
            layout=dict(width="auto"),
        )
        self._status = w.Label(layout=dict(width="auto"), style=dict(text_color="white"))
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._counter_total, self._status],
                    layout=dict(align_items="stretch", width="6em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self._hlayers = set()
        self.status = validate(self.puzzle)
        self.counters = count_finals(self.puzzle)

    def init_counter(self, dig: int):
        btn = w.Button(
            description=str(dig),
            button_style="info",
            layout=dict(width="auto"),
        )
        btn.on_click(lambda btn: self.toggle_layer(dig))
        return btn

    def toggle_layer(self, layer: int):
        if not self.puzzle:
            return
        if layer in self._hlayers:
            self._hlayers.remove(layer)
        else:
            self._hlayers.add(layer)
        self._canvas.draw_board(self.puzzle, self._hlayers)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "transparent"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            self._counters[dig].description = f"{dig} ({cnt})"
            self._counters[dig].style.background = "var(--jp-success-color0)" if cnt == 9 else ""
        total = counters.total()
        self._counter_total.value = f"Total: {total}"
        self._counter_total.style.background = "var(--jp-success-color0)" if total == 81 else ""

    @observe("targets", "finals", "anchors", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for trg in self.targets:
                self._highlight_target(trg, "orange")

            for trg in self.finals:
                self._highlight_target(trg, "pink")

            for lnk in self.links:
                self._highlight_link(lnk, "cyan")

            for trg in self.anchors:
                self._highlight_target(trg, "cyan")

    def _highlight_target(self, trg: Target, color: str):
        for loc in iter(trg.zone):
            self._canvas.highlight_segment(loc, trg.dig, color=color)

    def _highlight_link(self, lnk: Link, color: str):
        t1, t2 = lnk
        assert t1.zone.is_cellular and t2.zone.is_cellular, "cannon highlight group links"
        self._canvas.highlight_link(t1.zone.loc(), t1.dig, t2.zone.loc(), t2.dig, style=LINK_STYLES[lnk.__class__.__name__], color="blue")

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    def click_continue(self) -> asyncio.Future[bool]:
        return click_future(self._continue)

    async def pause(self):
        self.paused = True
        await self.click_continue()
        self.paused = False

In [ ]:
debug_view = w.Output()
gui = GUI()
display(gui, debug_view)

In [ ]:
gui.puzzle = puzzle

In [ ]:
async def solve_ui(initial: Board, *resolvers: Resolver):
    result = initial
    gui.puzzle = result
    gui.running = True
    gui.inspecting = {r.__name__: True for r in resolvers}

    try:
        iteration = 0
        async for resolver, resolution, result in solver(initial, *resolvers):
            iteration += 1
            print(iteration, resolver.__name__)
            # pprint(resolution)
            if gui.inspecting[resolver.__name__]:
                resolving = f"#{iteration} {resolver.__name__}: "
                if resolution.castaways:
                    resolving += f"-= {len(resolution.castaways)}"
                if resolution.finals:
                    resolving += f"== {len(resolution.finals)}"
                gui.resolving = resolving
                render_resolution(resolution)
                await gui.pause()
                clear_resolution()

                gui.puzzle = result
                await asyncio.sleep(0.2)
            else:
                gui.puzzle = result
            gui.resolving = f"#{iteration}"
    except Exception as e:
        with debug_view:
            raise RuntimeError("Solver failed") from e

    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Resolution):
    gui.targets = res.castaways if res.castaways else set()
    gui.finals = res.finals if res.finals else set()
    if res.highlights is not None:
        gui.anchors = res.highlights.get("anchors", set())
        if "chain" in res.highlights:
            gui.links = set(res.highlights["chain"])
        elif "links" in res.highlights:
            gui.links = res.highlights["links"]
        else:
            gui.links = set()


def clear_resolution():
    gui.targets = set()
    gui.finals = set()
    gui.anchors = set()
    gui.links = set()


In [ ]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        cleanup,
        singles,
        clean_mults_(2),
        unhide_mults_(2),
        clean_mults_(3),
        unhide_mults_(3),
        clean_mults_(4),
        unhide_mults_(4),
        clean_mults_(5),
        unhide_mults_(5),
        chains_("rope"),
        chains_("loop"),
    )
)

In [ ]:
task

In [ ]:
task.cancel()

In [ ]:
puzzle = task.result()